In [8]:
base = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'

for split in os.listdir(base):
    print(split, '->', os.listdir(os.path.join(base, split)))

Training -> ['pituitary', 'notumor', 'meningioma', 'glioma']
Testing -> ['pituitary', 'notumor', 'meningioma', 'glioma']


In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 256
BATCH_SIZE = 32
base = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15,
    rotation_range=5,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_ds = train_datagen.flow_from_directory(
    f'{base}/Training',
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='training',
    seed=42
)

val_ds = train_datagen.flow_from_directory(
    f'{base}/Training',
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='validation',
    seed=42
)

test_ds = test_datagen.flow_from_directory(
    f'{base}/Testing',
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = list(train_ds.class_indices.keys())
print(class_names)

Found 4760 images belonging to 4 classes.
Found 840 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
['glioma', 'meningioma', 'notumor', 'pituitary']


In [10]:
!pip install keras-tuner

In [11]:
import keras_tuner as kt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.metrics import Recall, Precision

def build_model(hp):
    model = Sequential()
    for i in range(hp.Int('layers', min_value=3, max_value=6)):
        model.add(Conv2D(
            hp.Int(f"neurons{i}", min_value=16, max_value=96, step=16),
            activation='relu',
            kernel_size=(3,3),
            padding='same',
            input_shape=(256,256,1) if i == 0 else None
        ))
        model.add(MaxPooling2D(pool_size=(2,2)))
        model.add(Dropout(hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)))

    model.add(Flatten())
    model.add(Dense(hp.Int("dense_units", min_value=32, max_value=128, step=32), activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(4, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy', Recall(name='recall'), Precision(name='precision')]
    )
    return model

In [12]:
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective('val_recall', direction='max'),
    max_trials=10,
    executions_per_trial=1,
    directory='new_tuner_v2',
    project_name='brain_tumor_detection_cnn_v2'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1790081095.163511      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790081095.166365      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [13]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', mode='min', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5, patience=3, min_lr=1e-6)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)

Trial 10 Complete [00h 06m 51s]
val_recall: 0.8214285969734192

Best val_recall So Far: 0.8928571343421936
Total elapsed time: 01h 07m 34s


In [14]:
train_ds

In [15]:
tuner.get_best_hyperparameters()[0].values

{'layers': 4,
 'neurons0': 80,
 'dropout_0': 0.1,
 'neurons1': 16,
 'dropout_1': 0.1,
 'neurons2': 48,
 'dropout_2': 0.30000000000000004,
 'dense_units': 64,
 'neurons3': 96,
 'dropout_3': 0.4,
 'neurons4': 96,
 'dropout_4': 0.4,
 'neurons5': 80,
 'dropout_5': 0.1}

In [16]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 80)   │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 80)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 128, 80)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 16)   │        11,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 48)     │         6,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32, 32, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 96)     │        41,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16, 16, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 24576)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,572,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,634,052 (6.23 MB)

 Trainable params: 1,634,052 (6.23 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
tuner.results_summary()

Results summary
Results in new_tuner_v2/brain_tumor_detection_cnn_v2
Showing 10 best trials
Objective(name="val_recall", direction="max")

Trial 04 summary
Hyperparameters:
layers: 4
neurons0: 80
dropout_0: 0.1
neurons1: 16
dropout_1: 0.1
neurons2: 48
dropout_2: 0.30000000000000004
dense_units: 64
neurons3: 96
dropout_3: 0.4
neurons4: 96
dropout_4: 0.4
neurons5: 80
dropout_5: 0.1
Score: 0.8928571343421936

Trial 08 summary
Hyperparameters:
layers: 3
neurons0: 32
dropout_0: 0.2
neurons1: 80
dropout_1: 0.1
neurons2: 16
dropout_2: 0.4
dense_units: 128
neurons3: 96
dropout_3: 0.1
neurons4: 64
dropout_4: 0.4
neurons5: 64
dropout_5: 0.4
Score: 0.8916666507720947

Trial 05 summary
Hyperparameters:
layers: 4
neurons0: 64
dropout_0: 0.1
neurons1: 32
dropout_1: 0.4
neurons2: 80
dropout_2: 0.30000000000000004
dense_units: 64
neurons3: 64
dropout_3: 0.2
neurons4: 80
dropout_4: 0.2
neurons5: 80
dropout_5: 0.30000000000000004
Score: 0.8809523582458496

Trial 07 summary
Hyperparameters:
layers: 3
neu

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', mode='min', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5, patience=3, min_lr=1e-6)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/40
149/149 ━━━━━━━━━━━━━━━━━━━━ 51s 298ms/step - accuracy: 0.9059 - loss: 0.2536 - precision: 0.9130 - recall: 0.8971 - val_accuracy: 0.9095 - val_loss: 0.2642 - val_precision: 0.9209 - val_recall: 0.9012 - learning_rate: 0.0010
Epoch 2/40
149/149 ━━━━━━━━━━━━━━━━━━━━ 38s 255ms/step - accuracy: 0.9174 - loss: 0.2241 - precision: 0.9236 - recall: 0.9118 - val_accuracy: 0.9143 - val_loss: 0.2691 - val_precision: 0.9216 - val_recall: 0.9095 - learning_rate: 0.0010
Epoch 3/40
149/149 ━━━━━━━━━━━━━━━━━━━━ 38s 255ms/step - accuracy: 0.9216 - loss: 0.2146 - precision: 0.9274 - recall: 0.9124 - val_accuracy: 0.9310 - val_loss: 0.2108 - val_precision: 0.9372 - val_recall: 0.9238 - learning_rate: 0.0010
Epoch 4/40
149/149 ━━━━━━━━━━━━━━━━━━━━ 38s 257ms/step - accuracy: 0.9191 - loss: 0.2118 - precision: 0.9243 - recall: 0.9134 - val_accuracy: 0.9286 - val_loss: 0.2164 - val_precision: 0.9370 - val_recall: 0.9202 - learning_rate: 0.0010
Epoch 5/40
149/149 ━━━━━━━━━━━━━━━━━━━━ 38s 253ms/st

In [20]:
results = model.evaluate(test_ds)
print(results)   # see the order first

50/50 ━━━━━━━━━━━━━━━━━━━━ 15s 309ms/step - accuracy: 0.9150 - loss: 1.1968 - precision: 0.9156 - recall: 0.9150
[1.1968246698379517, 0.9150000214576721, 0.9150000214576721, 0.9155722260475159]


In [21]:
test_loss, test_acc, test_recall, test_precision = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test Precision: {test_precision:.4f}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - accuracy: 0.9150 - loss: 1.1968 - precision: 0.9156 - recall: 0.9150
Test Loss: 1.1968
Test Accuracy: 0.9150
Test Recall: 0.9150
Test Precision: 0.9156


In [32]:
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_image(path, model, class_names):
    img = image.load_img(path, color_mode='grayscale', target_size=(256, 256))
    arr = image.img_to_array(img) / 255.   # must match rescale=1./255 above
    arr = np.expand_dims(arr, axis=0)
    pred = model.predict(arr, verbose=0)
    pred_class = class_names[np.argmax(pred)]
    confidence = np.max(pred) * 100
    return pred_class, confidence

pred_class, conf = predict_image("/kaggle/input/datasets/azrapatvi/testing/brain5.jpg", model, class_names)
print("Predicted:", pred_class, "| Confidence:", conf)

Predicted: notumor | Confidence: 100.0


In [24]:
import os
glioma_dir = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/glioma'
files_list = os.listdir(glioma_dir)[:9]
for f in files_list:
    path = os.path.join(glioma_dir, f)
    pred_class, conf = predict_image(path, model, class_names)
    print(f"{f}: Pred={pred_class} ({conf:.1f}%)")

Tr-gl_1362.jpg: Pred=glioma (100.0%)
Tr-gl_821.jpg: Pred=glioma (100.0%)
Tr-gl_825.jpg: Pred=glioma (100.0%)
Tr-gl_315.jpg: Pred=glioma (100.0%)
Tr-gl_52.jpg: Pred=glioma (100.0%)
Tr-gl_626.jpg: Pred=glioma (99.9%)
Tr-gl_827.jpg: Pred=glioma (100.0%)
Tr-gl_1033.jpg: Pred=glioma (91.2%)
Tr-gl_1096.jpg: Pred=glioma (100.0%)


In [25]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_true = test_ds.classes
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

      glioma       0.98      0.76      0.86       400
  meningioma       0.88      0.91      0.90       400
     notumor       0.85      1.00      0.92       400
   pituitary       0.98      0.99      0.99       400

    accuracy                           0.92      1600
   macro avg       0.92      0.92      0.91      1600
weighted avg       0.92      0.92      0.91      1600

[[304  46  49   1]
 [  5 363  24   8]
 [  0   0 400   0]
 [  1   2   0 397]]


In [34]:
model.save('brain_tumor_model_final1.keras')

In [28]:
from IPython.display import FileLink
FileLink('/kaggle/working/brain_tumor_model_final.keras')

/kaggle/working/brain_tumor_model_final.keras

In [33]:
from IPython.display import FileLink
FileLink('/kaggle/working/brain_tumor_model_final.keras')

/kaggle/working/brain_tumor_model_final.keras